## Change Path

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Adjust this path to where config.py is located
code_dir = Path("/cluster/work/boeva/lrabuzin/deepcast/src")  # or something relative like ../code
sys.path.append(str(code_dir))

## Compare coding regions

In [3]:
from data_preparation.coding_snps.exon_regions import generate_coding_regions

In [4]:
coding_regions = generate_coding_regions()

13:28:31 - data_preparation.coding_snps.exon_regions - DEBUG - Sequences that don't match chromosomes we are looking at (67): 
 ['NC_000023.10', 'NC_000024.9', 'NC_012920.1', 'NT_113889.1', 'NT_113923.1', 'NT_113961.1', 'NT_167208.1', 'NT_167209.1', 'NT_167210.1', 'NT_167211.1', 'NT_167212.1', 'NT_167213.1', 'NT_167214.1', 'NT_167215.1', 'NT_167216.1', 'NT_167217.1', 'NT_167218.1', 'NT_167219.1', 'NT_167220.1', 'NT_167221.1', 'NT_167222.1', 'NT_167223.1', 'NT_167224.1', 'NT_167225.1', 'NT_167226.1', 'NT_167227.1', 'NT_167228.1', 'NT_167229.1', 'NT_167230.1', 'NT_167231.1', 'NT_167232.1', 'NT_167233.1', 'NT_167234.1', 'NT_167235.1', 'NT_167236.1', 'NT_167237.1', 'NT_167238.1', 'NT_167239.1', 'NT_167240.1', 'NT_167241.1', 'NT_167242.1', 'NT_167243.1', 'NW_003571064.2', 'NW_003871098.1', 'NW_003871099.1', 'NW_003871100.1', 'NW_003871101.3', 'NW_003871102.1', 'NW_003871103.3', 'NW_004070877.1', 'NW_004070878.1', 'NW_004070879.1', 'NW_004070880.2', 'NW_004070881.1', 'NW_004070882.1', 'NW_00

In [5]:
coding_regions = coding_regions.drop_duplicates(subset=['chr', 'start', 'end'])

### Load Sophie's coding regions

In [6]:
import pandas as pd
from config import EXON_REGIONS_PATH

In [7]:
exon_snps_sophie = pd.read_csv(EXON_REGIONS_PATH.parent / 'exon_regions_v2_old.csv')

In [8]:
coding_regions_sophie = exon_snps_sophie.drop_duplicates(subset=['chr', 'start', 'end'])

### Compare lists

In [9]:
len(coding_regions)

293747

In [10]:
len(coding_regions_sophie)

269604

In [11]:
common = coding_regions.merge(coding_regions_sophie, how='inner')

269604 269604 293747

In [12]:
diff = coding_regions.merge(common, how='left', indicator=True)

In [13]:
diff

,chr,start,end,_merge
0,1,11874,12227,both
1,1,12613,12721,both
2,1,13221,14409,both
3,1,29321,29370,both
4,1,24738,24891,both
...,...,...,...,...
293742,21,291381,291416,left_only
293743,21,291241,291286,left_only
293744,21,304605,304719,left_only
293745,21,305562,305957,left_only


In [14]:
extra = diff[diff['_merge']=='left_only']

In [19]:
sub_dfs = [sub_df for _, sub_df in extra.groupby('chr')]


In [20]:
len(sub_dfs)

22

In [21]:
for chr, sub_df in extra.groupby('chr'):
    print(f"Chromosome {chr} has {len(sub_df)} extra exon regions")
    print(sub_df.head())

Chromosome 1 has 2080 extra exon regions
        chr   start     end     _merge
271611    1   15660   16348  left_only
271612    1   11109   11380  left_only
271613    1    8107    8159  left_only
271614    1    5292    5390  left_only
271615    1  468241  468683  left_only
Chromosome 2 has 222 extra exon regions
        chr  start   end     _merge
281640    2   6693  7538  left_only
281641    2   7746  8228  left_only
281642    2   8335  8567  left_only
281643    2   1105  1172  left_only
281644    2   3865  3963  left_only
Chromosome 3 has 275 extra exon regions
        chr  start   end     _merge
281674    3   1446  1530  left_only
281675    3   2373  5521  left_only
281676    3   6614  6759  left_only
281677    3   7531  7658  left_only
281678    3   9060  9201  left_only
Chromosome 4 has 235 extra exon regions
        chr  start    end     _merge
269604    4  88200  88375  left_only
269605    4  75488  75675  left_only
269606    4  75049  75288  left_only
269607    4  59869  59956

In [15]:
set(extra['chr'])

{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22}

### Manual reconstruction (debugging)

In [22]:
from config import GFF_FILE_PATH

columns = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

if Path(GFF_FILE_PATH).exists():
    gff_df = pd.read_csv(GFF_FILE_PATH, sep="\t", comment='#', header=None, names=columns)

In [33]:
import re

pattern = re.compile(r"chromosome=(\d+)")
chr_dfs: dict[object, tuple[int, pd.DataFrame]] = {}

for seqid, sub_df in gff_df.groupby('seqid'):
    match = pattern.search(sub_df.iloc[0]['attributes'])
    if match:
        chr = int(match.group(1))
        chr_dfs[seqid] = (chr, sub_df)

In [35]:
sequences = {chr:[] for chr in range(1, 23)}

for seqid, (chr, sub_df) in chr_dfs.items():
    sequences[chr].append((chr, len(sub_df), seqid))
    # print(sub_df.columns)
    # print(sub_df.iloc[0].loc['seqid'])
    # print(f"Chromosome {chr}: length {len(sub_df)} seqid {sub_df['seqid']}")

In [36]:
sequences

{1: [(1, 170441, 'NC_000001.10'),
  (1, 1, 'NT_113878.1'),
  (1, 67, 'NT_167207.1'),
  (1, 36, 'NW_003315903.1'),
  (1, 4, 'NW_003315904.1'),
  (1, 221, 'NW_003315905.1'),
  (1, 504, 'NW_003315906.1'),
  (1, 128, 'NW_003315907.1'),
  (1, 163, 'NW_003571030.1'),
  (1, 13179, 'NW_003871055.3'),
  (1, 643, 'NW_003871056.3'),
  (1, 831, 'NW_003871057.1'),
  (1, 4, 'NW_004070863.1'),
  (1, 7, 'NW_004070864.2'),
  (1, 10, 'NW_004070865.1')],
 2: [(2, 142057, 'NC_000002.11'),
  (2, 19, 'NW_003315908.1'),
  (2, 208, 'NW_003315909.1'),
  (2, 48, 'NW_003571031.1'),
  (2, 118, 'NW_003571032.1'),
  (2, 99, 'NW_003571033.2'),
  (2, 413, 'NW_004504299.1')],
 3: [(3, 126346, 'NC_000003.11'),
  (3, 683, 'NW_003315910.1'),
  (3, 75, 'NW_003315911.1'),
  (3, 222, 'NW_003315912.1'),
  (3, 22, 'NW_003315913.1'),
  (3, 23, 'NW_003871058.1'),
  (3, 531, 'NW_003871059.1'),
  (3, 168, 'NW_003871060.1'),
  (3, 479, 'NW_004775426.1')],
 4: [(4, 77460, 'NC_000004.11'),
  (4, 11, 'NT_113885.1'),
  (4, 29, 'NT_113

### Debugged version

In [47]:
from data_preparation.coding_snps.exon_regions import generate_coding_regions

exon_regions_new = generate_coding_regions()

14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113878.1 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113885.1 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113888.1 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113889.1 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113891.2 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113901.1 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113907.1 since it's not reference genome.
14:03:48 - data_preparation.coding_snps.exon_regions - DEBUG - Skipping sequence NT_113909.1 since it's not reference 

In [48]:
exon_regions_new = exon_regions_new.drop_duplicates(subset=['chr', 'start', 'end'])

In [49]:
len(exon_regions_new)

269604

## Test coding snp list generation

In [14]:
from data_preparation.coding_snps.generate_coding_snp_list import generate_coding_snps_list

coding_snps = generate_coding_snps_list()

16:37:39 - data_preparation.coding_snps.exon_regions - DEBUG - Skipped 272 sequences since they're not reference genome: 
 ['NT_113878.1', 'NT_113885.1', 'NT_113888.1', 'NT_113889.1', 'NT_113891.2', 'NT_113901.1', 'NT_113907.1', 'NT_113909.1', 'NT_113911.1', 'NT_113914.1', 'NT_113915.1', 'NT_113916.2', 'NT_113921.2', 'NT_113923.1', 'NT_113930.1', 'NT_113941.1', 'NT_113943.1', 'NT_113945.1', 'NT_113947.1', 'NT_113948.1', 'NT_113949.1', 'NT_113950.2', 'NT_113961.1', 'NT_167207.1', 'NT_167208.1', 'NT_167209.1', 'NT_167210.1', 'NT_167211.1', 'NT_167212.1', 'NT_167213.1', 'NT_167214.1', 'NT_167215.1', 'NT_167216.1', 'NT_167217.1', 'NT_167218.1', 'NT_167219.1', 'NT_167220.1', 'NT_167221.1', 'NT_167222.1', 'NT_167223.1', 'NT_167224.1', 'NT_167225.1', 'NT_167226.1', 'NT_167227.1', 'NT_167228.1', 'NT_167229.1', 'NT_167230.1', 'NT_167231.1', 'NT_167232.1', 'NT_167233.1', 'NT_167234.1', 'NT_167235.1', 'NT_167236.1', 'NT_167237.1', 'NT_167238.1', 'NT_167239.1', 'NT_167240.1', 'NT_167241.1', 'NT_16

In [16]:
len(coding_snps)

749317

In [19]:
coding_snps.to_csv(EXON_REGIONS_PATH.parent / 'coding_snps.csv')

### Load Sophie's coding snps

In [5]:
import pandas as pd
from config import EXON_REGIONS_PATH

In [6]:
coding_snps_sophie = pd.read_csv(EXON_REGIONS_PATH)

In [7]:
coding_snps_sophie.duplicated().sum()

np.int64(470)

In [8]:
coding_snps_sophie = coding_snps_sophie.drop_duplicates(subset=['snp'])

In [9]:
len(coding_snps_sophie)

413088

### Compare lists

In [17]:
common = coding_snps.merge(coding_snps_sophie, how='inner')

In [18]:
len(common)

413088

In [93]:
diff = coding_snps.merge(common, how='left', indicator=True)

In [94]:
diff_sophie = coding_snps_sophie.merge(common, how='left', indicator=True)

In [95]:
extra = diff[diff['_merge']=='left_only']

In [96]:
extra_sophie = diff_sophie[diff_sophie['_merge']=='left_only']

In [97]:
extra

,snp,_merge
16,rs144015209,left_only
32,rs546055875,left_only
45,rs200450385,left_only
47,rs184843908,left_only
48,rs189795883,left_only
...,...,...
505209,rs369357973,left_only
505211,rs61143038,left_only
505213,rs78671094,left_only
505215,rs137886366,left_only


In [98]:
extra_sophie

,snp,_merge
327252,rs3747243,left_only
327253,rs78060012,left_only
327254,rs9616125,left_only
327255,rs57234197,left_only
327256,rs58819475,left_only
...,...,...
328102,rs12628964,left_only
328103,rs117477158,left_only
328104,rs9617018,left_only
328105,rs9617066,left_only


In [114]:
extra_sophie.isna().sum()

snp       1
_merge    0
dtype: int64

In [118]:
extra_sophie.duplicated().sum()

np.int64(0)

In [117]:
extra.isna().sum()

snp       0
_merge    0
dtype: int64

## Deconstruct coding snp list generation

In [68]:

from data_preparation.coding_snps.rs_ids import merge_sumstats_with_reference

coding_regions = generate_coding_regions().drop_duplicates(subset=['chr', 'start', 'end'])

merged_sumstats = merge_sumstats_with_reference('phecode-772.1-both_sexes.tsv.bgz')

14:27:09 - data_preparation.coding_snps.exon_regions - DEBUG - Skipped 272 sequences since they're not reference genome: 
 ['NT_113878.1', 'NT_113885.1', 'NT_113888.1', 'NT_113889.1', 'NT_113891.2', 'NT_113901.1', 'NT_113907.1', 'NT_113909.1', 'NT_113911.1', 'NT_113914.1', 'NT_113915.1', 'NT_113916.2', 'NT_113921.2', 'NT_113923.1', 'NT_113930.1', 'NT_113941.1', 'NT_113943.1', 'NT_113945.1', 'NT_113947.1', 'NT_113948.1', 'NT_113949.1', 'NT_113950.2', 'NT_113961.1', 'NT_167207.1', 'NT_167208.1', 'NT_167209.1', 'NT_167210.1', 'NT_167211.1', 'NT_167212.1', 'NT_167213.1', 'NT_167214.1', 'NT_167215.1', 'NT_167216.1', 'NT_167217.1', 'NT_167218.1', 'NT_167219.1', 'NT_167220.1', 'NT_167221.1', 'NT_167222.1', 'NT_167223.1', 'NT_167224.1', 'NT_167225.1', 'NT_167226.1', 'NT_167227.1', 'NT_167228.1', 'NT_167229.1', 'NT_167230.1', 'NT_167231.1', 'NT_167232.1', 'NT_167233.1', 'NT_167234.1', 'NT_167235.1', 'NT_167236.1', 'NT_167237.1', 'NT_167238.1', 'NT_167239.1', 'NT_167240.1', 'NT_167241.1', 'NT_16

In [119]:
merged_sumstats.isna().sum()

chr                        0
pos                        0
ref                        0
alt                        0
neglog10_pval_EUR    4975921
snp                  9193506
dtype: int64

In [122]:
merged_sumstats.dropna(subset=['snp']).isna().sum()

chr                        0
pos                        0
ref                        0
alt                        0
neglog10_pval_EUR    3145449
snp                        0
dtype: int64

In [123]:
extra_sophie.merge(merged_sumstats.dropna(subset=['snp']), how='left', on='snp') # [['snp', 'chr', 'pos']]

,snp,_merge,chr,pos,ref,alt,neglog10_pval_EUR
0,rs3747243,left_only,NaN,NaN,NaN,NaN,NaN
1,rs78060012,left_only,NaN,NaN,NaN,NaN,NaN
2,rs9616125,left_only,NaN,NaN,NaN,NaN,NaN
3,rs57234197,left_only,NaN,NaN,NaN,NaN,NaN
4,rs58819475,left_only,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
850,rs12628964,left_only,NaN,NaN,NaN,NaN,NaN
851,rs117477158,left_only,NaN,NaN,NaN,NaN,NaN
852,rs9617018,left_only,NaN,NaN,NaN,NaN,NaN
853,rs9617066,left_only,NaN,NaN,NaN,NaN,NaN


In [124]:
extra.merge(merged_sumstats.dropna(subset=['snp']), how='left', on='snp') #[['snp', 'pos']]

,snp,_merge,chr,pos,ref,alt,neglog10_pval_EUR
0,rs144015209,left_only,1,762187,C,T,NaN
1,rs546055875,left_only,1,777516,C,T,NaN
2,rs200450385,left_only,1,792601,T,C,NaN
3,rs184843908,left_only,1,793470,G,A,NaN
4,rs189795883,left_only,1,793553,T,A,NaN
...,...,...,...,...,...,...,...
93019,rs369357973,left_only,22,46405453,T,C,NaN
93020,rs61143038,left_only,22,46405761,C,T,NaN
93021,rs78671094,left_only,22,46405845,C,T,NaN
93022,rs137886366,left_only,22,46406142,C,G,NaN


In [ ]:
from data_preparation.coding_snps.in_exon_region import find_coding_snps

coding_snps = find_coding_snps(merged_sumstats, coding_regions)

Process ForkProcess-651:
Process ForkProcess-646:
Process ForkProcess-648:
Process ForkProcess-647:
Process ForkProcess-649:
Traceback (most recent call last):
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/concurrent/futures/process.py", line 249, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^


KeyboardInterrupt: 

  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/queues.py", line 102, in get
    with self._rlock:
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_g

  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/concurrent/futures/process.py", line 249, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
KeyboardInterrupt
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/cluster/apps/biomed/boeva/lrabuzin/conda/envs/deepcast_gwas/lib/python3.11/multiprocessing/queues.py", line 103, in get
    res = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
Process ForkProcess-650:
  File "/cl